# Prędkość odsypu a czas trwania

Czy prędkość odsypu zależy od czasu trwania (np. krótkie odsypy szybsze)? Zbiór jak w `event_stats.ipynb`: baza `2026-10-18 obiekt`, sesja 9, wariant `CURRENT_VARIANT`, odsypy przyjęte i bez kolizji, obie metody wskazań przed i po (`METHODS`: mediana 3 s, najbliższa próbka). Masa i prędkość jako wartości dodatnie (odsypany materiał); obliczenia w `dms_analysis.dispense_rate`.

Kontekst obiektu (wg użytkownika): odsyp kończy sterownik, gdy waga cementu osiągnie zadaną masę (np. 500 kg), więc czas trwania jest skutkiem prędkości wypływu, a nie odwrotnie.

Wybór zdarzeń, wspólny dla obu metod (zdarzenie odrzucone w jednej metodzie wypada z obu):

1. odsypy z różnicą ≥ 0 w którejkolwiek metodzie (sprzeczne z logiką),
2. opcjonalne przycięcie percentylami różnicy w każdej metodzie: `LOWER_TRIM_PERCENT` odcina najbardziej ujemne różnice (największe odsypy), `UPPER_TRIM_PERCENT` najbliższe zera (najmniejsze); domyślnie 0 = bez przycięcia.

Miary:

- **Pearson** — zależność liniowa prędkości od czasu trwania, wrażliwa na skrajne punkty,
- **Spearman** — to samo na rangach, mierzy zależność monotoniczną, odporny na skrajne punkty,
- **prosta masa(czas)** (najmniejsze kwadraty) — nachylenie = prędkość niezależna od stałego przesunięcia czasu; przesunięcie = miejsce przecięcia prostej z osią czasu (> 0: sygnał przed wypływem materiału, < 0: materiał jeszcze po końcu sygnału). Prędkość liczona jako masa / czas przy przesunięciu ≠ 0 daje pozorną zależność od czasu trwania. Ta interpretacja zakłada jedną stałą prędkość; przy dozowaniu do zadanej masy prędkość się zmienia, a masy skupiają się wokół wartości zadanych niezależnie od czasu, więc prosta wychodzi płaska i przesunięcie nie ma sensu fizycznego.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import display

from dms_analysis.dispense_rate import (
    box_figure,
    correlations,
    dispenses,
    full_refills,
    mass_figure,
    rate_figure,
    scatter_figure,
    trim_sensitivity,
    trimmed,
    with_context,
    without_non_negative,
    x_correlations,
)
from dms_analysis.event_stats import event_differences
from dms_analysis.prepared import load_session
from dms_analysis.signals import CURRENT_VARIANT
from dms_analysis.snapshot import ANALYSIS_SESSION, ANALYSIS_SOURCE
from dms_analysis.tables import collapsible

DB_PATH = Path("../data/prepared/obiekt-sesja-9.db")
FIGURE_WIDTH = 1200
LOWER_TRIM_PERCENT = 0
UPPER_TRIM_PERCENT = 0
SENSITIVITY_LEVELS = [0, 1, 5]
FULL_REFILL_MIN_KG = 10_000
CONTENT_BIN_T = 5
HOURS_SINCE_REFILL_BIN_H = 24
CLOCK_OFFSET_H = -4

pd.set_option("display.max_rows", 500)

In [ ]:
session = load_session(DB_PATH, ANALYSIS_SOURCE, ANALYSIS_SESSION, CURRENT_VARIANT)
events = event_differences(session)
all_dispenses = dispenses(events)
valid = without_non_negative(all_dispenses)
selection = trimmed(valid.kept, LOWER_TRIM_PERCENT, UPPER_TRIM_PERCENT)
dispense = selection.kept

count = all_dispenses["start"].nunique
print(f"odsypy: {count()}, odrzucone z różnicą ≥ 0: {valid.rejected['start'].nunique()}")
print(f"przycięcie dół {LOWER_TRIM_PERCENT}%, góra {UPPER_TRIM_PERCENT}%: odrzucone {selection.rejected['start'].nunique()}")
print(f"do analizy: {dispense['start'].nunique()}")

columns = ["start", "duration_s", "method", "difference_kg"]
display(collapsible(valid.rejected[columns].sort_values("start", kind="stable"), "Odrzucone z różnicą ≥ 0", opened=True))
display(collapsible(selection.rejected[columns].sort_values("start", kind="stable"), "Odrzucone przez przycięcie"))

## Korelacja i prosta masa(czas)

In [ ]:
correlations(dispense).round(3)

In [ ]:
display(rate_figure(dispense, "Średnia prędkość odsypu a czas trwania", FIGURE_WIDTH))
display(mass_figure(dispense, "Odsypana masa a czas trwania", FIGURE_WIDTH))

## Prędkość odsypu a zawartość silosu

Czy prędkość odsypu zależy od tego, ile materiału jest w silosie? Zawartość = wskazanie przed odsypem (`before_kg` danej metody), w tonach. Te same odsypy co wyżej (po odrzuceniu różnic ≥ 0 i przycięciu).

Zawartość nie jest niezależna od czasu: po zasypie silos opróżnia się przez kilka dni, więc wysoka zawartość to zawsze „krótko po zasypie”. Dlatego punkty są też pokolorowane okresami między pełnymi zasypami (zasyp z różnicą ≥ `FULL_REFILL_MIN_KG` w metodzie najbliższej próbki; krótkie sygnały zasypu bez przyrostu wagi nie liczą się): jeśli zależność wynika z zawartości, powinna być podobna w każdym okresie.

Miary jak wyżej (Pearson, Spearman) i nachylenie prostej na tonę zawartości, dla prędkości oraz dla jej składników: czasu trwania i odsypanej masy (prędkość = masa / czas).

In [ ]:
refills = full_refills(events, FULL_REFILL_MIN_KG)
context = with_context(dispense, refills, CLOCK_OFFSET_H)

display(refills)
display(context[context["method"] == "mediana 3 s"].groupby("period").size().rename("odsypy").to_frame())
x_correlations(context, "content_t").round(3)

In [ ]:
display(scatter_figure(context, "content_t", "dispense_rate_kg_s", "Średnia prędkość odsypu a zawartość silosu", FIGURE_WIDTH))
display(scatter_figure(context, "content_t", "dispense_rate_kg_s", "Średnia prędkość odsypu a zawartość silosu, okresy między zasypami", FIGURE_WIDTH, color="period"))

### Rozkład prędkości w przedziałach zawartości

Wykres pudełkowy dla przedziałów zawartości co `CONTENT_BIN_T` t: linia w pudełku = mediana, pudełko = kwartyle, wąsy = zakres bez punktów odstających; obok wszystkie odsypy z przedziału. Przy kilku punktach w przedziale pudełko niewiele mówi.

In [ ]:
display(box_figure(context, "content_t", "dispense_rate_kg_s", CONTENT_BIN_T, "Średnia prędkość odsypu w przedziałach zawartości", FIGURE_WIDTH))

### Czas trwania i masa a zawartość

Która składowa prędkości zmienia się z zawartością: czas trwania (wolniejszy wypływ) czy odsypana masa (inne porcje)?

In [ ]:
display(scatter_figure(context, "content_t", "duration_s", "Czas trwania odsypu a zawartość silosu", FIGURE_WIDTH, color="period"))
display(scatter_figure(context, "content_t", "dispensed_kg", "Odsypana masa a zawartość silosu", FIGURE_WIDTH, color="period"))

## Prędkość odsypu a czas od ostatniego zasypu

Czy znaczenie ma stan materiału w silosie (np. ubicie albo temperatura świeżo zasypanego cementu), a nie sama ilość? Czas liczony od końca ostatniego pełnego zasypu (jak wyżej) do początku odsypu. Odsypy sprzed pierwszego pełnego zasypu w sesji nie mają tej wartości i są pominięte.

Uwaga: w obrębie jednego okresu czas od zasypu i zawartość zmieniają się razem (silos się opróżnia), więc same te dane nie rozdzielą obu wpływów; różnią się dopiero między okresami.

In [ ]:
complete = context[context["method"] == "mediana 3 s"].dropna(subset=["hours_since_refill"])
print(f"pominięte bez wcześniejszego zasypu: {context['hours_since_refill'].isna().sum() // 2}")
print("korelacja zawartości z czasem od zasypu w okresach (Pearson):")
display(complete.groupby("period")[["content_t", "hours_since_refill"]].corr().xs("content_t", level=1)["hours_since_refill"].round(3).to_frame())
x_correlations(context, "hours_since_refill").round(3)

In [ ]:
display(scatter_figure(context, "hours_since_refill", "dispense_rate_kg_s", "Średnia prędkość odsypu a czas od ostatniego zasypu", FIGURE_WIDTH, color="period"))
display(box_figure(context, "hours_since_refill", "dispense_rate_kg_s", HOURS_SINCE_REFILL_BIN_H, "Średnia prędkość odsypu a czas od ostatniego zasypu, przedziały", FIGURE_WIDTH))

## Prędkość odsypu a godzina dnia

Czy rozrzut prędkości zależy od pory dnia (np. największy koło południa)? Godzina z zegara RPi przesunięta o `CLOCK_OFFSET_H` godzin. RPi nie ma RTC, więc przesunięcie jest przybliżone: według użytkownika godzina 16 na zegarze RPi to w rzeczywistości ok. 12–13. Przesunięcie zmienia tylko podpisy osi, nie kształt rozkładu. Przedziały co godzinę; przy kilku odsypach w godzinie pudełko niewiele mówi.

In [ ]:
display(scatter_figure(context, "hour_of_day", "dispense_rate_kg_s", "Średnia prędkość odsypu a godzina dnia", FIGURE_WIDTH, color="period", trend=False))
display(box_figure(context, "hour_of_day", "dispense_rate_kg_s", 1, "Średnia prędkość odsypu w godzinach dnia", FIGURE_WIDTH))

## Wrażliwość na przycięcie

Te same miary dla każdej kombinacji przycięcia dołu i góry z `SENSITIVITY_LEVELS` (w procentach), liczone na odsypach po odrzuceniu różnic ≥ 0. Jeśli wnioski się nie zmieniają, nie trzeba wybierać poziomu przycięcia.

In [ ]:
display(collapsible(trim_sensitivity(valid.kept, SENSITIVITY_LEVELS).round(3), "Wrażliwość na przycięcie", opened=True))